In [1]:

import pandas as pd

df = pd.read_csv("demandsense_synthetic_hotel_demand.csv")



In [2]:



print("First 5 rows:")
print(df.head())

print("\nDataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns)

print("\nDataset information:")
df.info()


First 5 rows:
         date hotel_id           hotel_name  total_rooms  available_rooms  \
0  2023-01-01     H001  Mountain View Hotel           80               80   
1  2023-01-02     H001  Mountain View Hotel           80               80   
2  2023-01-03     H001  Mountain View Hotel           80               80   
3  2023-01-04     H001  Mountain View Hotel           80               80   
4  2023-01-05     H001  Mountain View Hotel           80               78   

   occupied_rooms  occupancy_rate  bookings  bookings_cancelled  avg_price  \
0              54           67.50        36                   7    4747.33   
1              38           47.50        41                   5    4701.70   
2              37           46.25        41                   3    4172.83   
3              36           45.00        34                   2    4253.72   
4              45           57.69        38                   2    4290.55   

  day_of_week  month  is_weekend  is_holiday festival_

In [3]:

# Check missing values
print("Missing values:")
print(df.isnull().sum())

# Check duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

# Check duplicate hotel-date records
print("Duplicate hotel-date records:",
      df.duplicated(subset=["hotel_id", "date"]).sum())

# Check occupancy range
print("\nOccupancy rate range:")
print(df["occupancy_rate"].min(), "to",
      df["occupancy_rate"].max())


Missing values:
date                     0
hotel_id                 0
hotel_name               0
total_rooms              0
available_rooms          0
occupied_rooms           0
occupancy_rate           0
bookings                 0
bookings_cancelled       0
avg_price                0
day_of_week              0
month                    0
is_weekend               0
is_holiday               0
festival_or_event     5460
weather                  0
tourist_arrivals         0
data_type                0
dtype: int64

Duplicate rows: 0
Duplicate hotel-date records: 0

Occupancy rate range:
17.78 to 98.33


In [4]:

# Replace missing festival/event values
df["festival_or_event"] = df["festival_or_event"].fillna("No event")

# Check missing values again
print(df.isnull().sum())

# Check festival/event categories
print("\nFestival or event values:")
print(df["festival_or_event"].value_counts())


date                  0
hotel_id              0
hotel_name            0
total_rooms           0
available_rooms       0
occupied_rooms        0
occupancy_rate        0
bookings              0
bookings_cancelled    0
avg_price             0
day_of_week           0
month                 0
is_weekend            0
is_holiday            0
festival_or_event     0
weather               0
tourist_arrivals      0
data_type             0
dtype: int64

Festival or event values:
festival_or_event
No event                 5460
Summer Holiday Season     940
Diwali Period             285
Christmas Period          120
New Year                   40
Holi                       40
Independence Day           20
Name: count, dtype: int64


In [5]:

df["date"] = pd.to_datetime(df["date"])

print(df["date"].dtype)
print(df[["date", "hotel_id"]].head())


datetime64[ns]
        date hotel_id
0 2023-01-01     H001
1 2023-01-02     H001
2 2023-01-03     H001
3 2023-01-04     H001
4 2023-01-05     H001


In [6]:

# Check if occupied rooms exceed available rooms
print("Occupied rooms greater than available rooms:")
print((df["occupied_rooms"] > df["available_rooms"]).sum())

# Calculate occupancy rate again
df["calculated_occupancy"] = (
    df["occupied_rooms"] / df["available_rooms"]
) * 100

# Compare calculated and existing occupancy
difference = abs(
    df["occupancy_rate"] - df["calculated_occupancy"]
)

print("\nMismatched occupancy records:", (difference > 0.02).sum())

# Remove temporary column
df.drop("calculated_occupancy", axis=1, inplace=True)


Occupied rooms greater than available rooms:
0

Mismatched occupancy records: 0


In [9]:

# Validate and standardize room occupancy data

import pandas as pd

# Work on a copy of the original data
clean_df = df.copy()

# Convert columns to numeric
for column in ["total_rooms", "occupied_rooms"]:
    clean_df[column] = pd.to_numeric(
        clean_df[column], errors="raise"
    )

# Validate the source room counts
invalid = (
    (clean_df["total_rooms"] <= 0)
    | (clean_df["occupied_rooms"] < 0)
    | (clean_df["occupied_rooms"] > clean_df["total_rooms"])
)

print("Invalid room-count rows:", invalid.sum())

if invalid.any():
    raise ValueError(
        "Invalid room counts. Investigate before continuing."
    )

# Recalculate fields derived from room capacity and occupancy
clean_df["available_rooms"] = (
    clean_df["total_rooms"] - clean_df["occupied_rooms"]
)

clean_df["occupancy_rate"] = (
    clean_df["occupied_rooms"]
    / clean_df["total_rooms"] * 100
).round(2)

# Save the validated dataset
clean_df.to_csv("demandsense_consistent.csv", index=False)

print("Validated dataset saved.")
print("Rows:", len(clean_df))
print("Hotels:", clean_df["hotel_id"].nunique())


Invalid room-count rows: 0
Validated dataset saved.
Rows: 6905
Hotels: 5


In [7]:

df["year"] = df["date"].dt.year
df["day"] = df["date"].dt.day
df["day_of_year"] = df["date"].dt.dayofyear

print(df[["date", "year", "month", "day", "day_of_year"]].head())


        date  year  month  day  day_of_year
0 2023-01-01  2023      1    1            1
1 2023-01-02  2023      1    2            2
2 2023-01-03  2023      1    3            3
3 2023-01-04  2023      1    4            4
4 2023-01-05  2023      1    5            5


In [8]:
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum().sum())

print("\nDuplicate records:", df.duplicated().sum())

print("\nColumns:")
print(df.columns.tolist())

# Save the final cleaned dataset
df.to_csv("demandsense_cleaned.csv", index=False)

print("\nCleaned dataset saved successfully!")

Dataset shape: (6905, 21)

Missing values:
0

Duplicate records: 0

Columns:
['date', 'hotel_id', 'hotel_name', 'total_rooms', 'available_rooms', 'occupied_rooms', 'occupancy_rate', 'bookings', 'bookings_cancelled', 'avg_price', 'day_of_week', 'month', 'is_weekend', 'is_holiday', 'festival_or_event', 'weather', 'tourist_arrivals', 'data_type', 'year', 'day', 'day_of_year']

Cleaned dataset saved successfully!
